# 1. Feedforward Neural Networks (FNN): data flows in one direction (input -> output)

Theory Explanation:
Feedforward Neural Networks are the simplest type of artificial neural network
Architecture: Input Layer -> Hidden Layer(s) -> Output Layer
Information flows in one direction only (no cycles or loops)
Each layer is fully connected to the next layer
Mathematical representation: y = f(W_n * f(W_{n-1} * ... * f(W_1 * x + b_1) + b_{n-1}) + b_n)
Where W_i are weight matrices, b_i are bias vectors, and f is the activation function

Production Ready FNN Model Pipeline:

In [ ]:


import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import joblib
import logging

class ProductionFNN:
    def __init__(self, input_dim, hidden_layers=[128, 64, 32], output_dim=1, 
                 activation='relu', output_activation='sigmoid', dropout_rate=0.3):
        self.input_dim = input_dim
        self.hidden_layers = hidden_layers
        self.output_dim = output_dim
        self.activation = activation
        self.output_activation = output_activation
        self.dropout_rate = dropout_rate
        self.model = None
        self.scaler = StandardScaler()
        self.label_encoder = LabelEncoder()
        
    def build_model(self):
        """Build the FNN architecture"""
        self.model = Sequential()
        
        # Input layer
        self.model.add(Dense(self.hidden_layers[0], 
                           input_dim=self.input_dim, 
                           activation=self.activation))
        self.model.add(BatchNormalization())
        self.model.add(Dropout(self.dropout_rate))
        
        # Hidden layers
        for units in self.hidden_layers[1:]:
            self.model.add(Dense(units, activation=self.activation))
            self.model.add(BatchNormalization())
            self.model.add(Dropout(self.dropout_rate))
        
        # Output layer
        self.model.add(Dense(self.output_dim, activation=self.output_activation))
        
        return self.model
    
    def preprocess_data(self, X, y=None, fit_transform=True):
        """Preprocess input data"""
        if fit_transform:
            X_scaled = self.scaler.fit_transform(X)
            if y is not None:
                if y.dtype == 'object':
                    y_encoded = self.label_encoder.fit_transform(y)
                else:
                    y_encoded = y
                return X_scaled, y_encoded
            return X_scaled
        else:
            X_scaled = self.scaler.transform(X)
            if y is not None:
                if y.dtype == 'object':
                    y_encoded = self.label_encoder.transform(y)
                else:
                    y_encoded = y
                return X_scaled, y_encoded
            return X_scaled
    
    def train(self, X_train, y_train, X_val=None, y_val=None, 
              epochs=100, batch_size=32, learning_rate=0.001):
        """Train the FNN model"""
        # Preprocess data
        X_train_scaled, y_train_encoded = self.preprocess_data(X_train, y_train, fit_transform=True)
        
        if X_val is not None and y_val is not None:
            X_val_scaled, y_val_encoded = self.preprocess_data(X_val, y_val, fit_transform=False)
            validation_data = (X_val_scaled, y_val_encoded)
        else:
            validation_data = None
        
        # Build model if not already built
        if self.model is None:
            self.build_model()
        
        # Compile model
        optimizer = Adam(learning_rate=learning_rate)
        if self.output_dim == 1:
            loss = 'binary_crossentropy'
            metrics = ['accuracy']
        else:
            loss = 'sparse_categorical_crossentropy'
            metrics = ['accuracy']
        
        self.model.compile(optimizer=optimizer, loss=loss, metrics=metrics)
        
        # Callbacks for production training
        callbacks = [
            EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7)
        ]
        
        # Train model
        history = self.model.fit(
            X_train_scaled, y_train_encoded,
            epochs=epochs,
            batch_size=batch_size,
            validation_data=validation_data,
            callbacks=callbacks,
            verbose=1
        )
        
        return history
    
    def predict(self, X):
        """Make predictions"""
        X_scaled = self.preprocess_data(X, fit_transform=False)
        predictions = self.model.predict(X_scaled)
        
        if self.output_dim == 1:
            return (predictions > 0.5).astype(int).flatten()
        else:
            return np.argmax(predictions, axis=1)
    
    def predict_proba(self, X):
        """Get prediction probabilities"""
        X_scaled = self.preprocess_data(X, fit_transform=False)
        return self.model.predict(X_scaled)
    
    def evaluate(self, X_test, y_test):
        """Evaluate model performance"""
        X_test_scaled, y_test_encoded = self.preprocess_data(X_test, y_test, fit_transform=False)
        
        # Get predictions
        predictions = self.predict(X_test)
        
        # Calculate metrics
        accuracy = accuracy_score(y_test_encoded, predictions)
        report = classification_report(y_test_encoded, predictions)
        
        return accuracy, report
    
    def save_model(self, model_path, scaler_path, encoder_path=None):
        """Save trained model and preprocessors"""
        self.model.save(model_path)
        joblib.dump(self.scaler, scaler_path)
        if encoder_path and hasattr(self, 'label_encoder'):
            joblib.dump(self.label_encoder, encoder_path)
    
    def load_model(self, model_path, scaler_path, encoder_path=None):
        """Load trained model and preprocessors"""
        self.model = tf.keras.models.load_model(model_path)
        self.scaler = joblib.load(scaler_path)
        if encoder_path:
            self.label_encoder = joblib.load(encoder_path)

# Example usage for production deployment
def train_production_fnn(data_path, target_column, test_size=0.2, random_state=42):
    """Complete pipeline for training production FNN"""
    
    # Load and prepare data
    df = pd.read_csv(data_path)
    X = df.drop(columns=[target_column])
    y = df[target_column]
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=random_state, stratify=y_train
    )
    
    # Initialize and train model
    fnn = ProductionFNN(
        input_dim=X_train.shape[1],
        hidden_layers=[128, 64, 32],
        output_dim=len(np.unique(y)) if len(np.unique(y)) > 2 else 1
    )
    
    # Train model
    history = fnn.train(X_train, y_train, X_val, y_val, epochs=100, batch_size=32)
    
    # Evaluate model
    accuracy, report = fnn.evaluate(X_test, y_test)
    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"Classification Report:\n{report}")
    
    # Save model for production
    fnn.save_model('fnn_model.h5', 'scaler.pkl', 'label_encoder.pkl')
    
    return fnn, history

# Production inference pipeline
class FNNInferenceService:
    def __init__(self, model_path, scaler_path, encoder_path=None):
        self.fnn = ProductionFNN(input_dim=None)
        self.fnn.load_model(model_path, scaler_path, encoder_path)
    
    def predict_single(self, features):
        """Predict single instance"""
        features_array = np.array(features).reshape(1, -1)
        return self.fnn.predict(features_array)[0]
    
    def predict_batch(self, features_list):
        """Predict batch of instances"""
        features_array = np.array(features_list)
        return self.fnn.predict(features_array)
    
    def get_prediction_confidence(self, features):
        """Get prediction with confidence scores"""
        features_array = np.array(features).reshape(1, -1)
        probabilities = self.fnn.predict_proba(features_array)[0]
        prediction = self.fnn.predict(features_array)[0]
        confidence = np.max(probabilities)
        return prediction, confidence






# Key advantages of FNN:
1. Simple architecture - easy to understand and implement
2. Universal function approximator - can learn any continuous function
3. Fast training and inference compared to more complex architectures
4. Good baseline model for many machine learning tasks
5. Works well with tabular data and structured problems

# Production considerations:
1. Data preprocessing and normalization
2. Regularization techniques (dropout, batch normalization)
3. Model monitoring and retraining pipelines
4. A/B testing for model deployment
5. Scalable inference infrastructure
6. Model versioning and rollback capabilities